## Acesso ao Lakehouse

In [ ]:
import struct
import pandas as pd
import pyodbc
from azure.identity import InteractiveBrowserCredential

# ── Parâmetros ────────────────────────────────────────────────────────────────
# server é o endpoint para SQL copiado das Configurações do lakehouse (a partir dos 3 pontinhos)
SERVER    = "beu5bmmdbuwedpv62ucm524jzi-dmrv7k3fbwbevh5d4sidg3urfq.datawarehouse.fabric.microsoft.com"

In [ ]:
LAKEHOUSE = "lake_prep_stat"
SCHEMA    = "dbo"
LIMIT    = None   # linhas para inspeção; None = sem limite (cuidado com tabelas grandes)

In [54]:
# buscando as tabelas silver de stat
LAKEHOUSE = "lake_prep_stat"
SCHEMA    = "dbo"
TABELA    = "raw_rps_pessoas" # todo o histórico
LIMIT    = None   # linhas para inspeção; None = sem limite (cuidado com tabelas grandes)

In [4]:
# Tabela a ser consumida (vale a última)
TABELA    = "raw_siplan_lancamento_mensurador" # todo o histórico

In [ ]:
# Tabela a ser consumida (vale a última)
TABELA    = "raw_acoes_all" # todo o histórico

In [18]:
# Tabela a ser consumida (vale a última)
TABELA    = "raw_acoes" # do ano

## Conecta ao lakehouse e carrega a tabela indicada

In [6]:
print(f"Conectando ao Lakehouse {LAKEHOUSE}...")

credential = InteractiveBrowserCredential()
token = credential.get_token("https://analysis.windows.net/powerbi/api/.default")
token_bytes = token.token.encode("utf-16-le")
tokenstruct = struct.pack("<i", len(token_bytes)) + token_bytes
SQL_COPT_SS_ACCESS_TOKEN = 1256

conn_str = (
    f"Driver={{ODBC Driver 18 for SQL Server}};"
    f"Server={SERVER},1433;"
    f"Database={LAKEHOUSE};"
    "Encrypt=Yes;TrustServerCertificate=No;"
)
conn = pyodbc.connect(conn_str, attrs_before={SQL_COPT_SS_ACCESS_TOKEN: tokenstruct})

_top = f"TOP {LIMIT}" if LIMIT else ""
df = pd.read_sql(
    f"SELECT {_top} * FROM {SCHEMA}.{TABELA}"
    " WHERE [siplan_lancamento_mensurador.valor_mensurador] >= 0 AND [siplan_lancamento_mensurador.ano_sessao] = 2024",
    conn
)

# Remove prefixo "siplan_lancamento_mensurador." nos nomes de colunas gravado pelo Dataflow
df.columns = [c.replace("siplan_lancamento_mensurador.", "") for c in df.columns]

print(f"Registros carregados : {len(df):>10,}")
print(f"Colunas              : {list(df.columns)}")
df.head()

Conectando ao Lakehouse lake_prep_stat...


C:\Users\INTEL\AppData\Local\Temp\ipykernel_28948\3003557819.py:18: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(


Registros carregados :  3,271,579
Colunas              : ['uo', 'uor_sigla', 'atividade_id', 'atividade_nome', 'atividade_complemento', 'atividade_status', 'estatistico_status', 'projeto_id', 'projeto_nome', 'gratuidade', 'gestor_atividade_integ', 'data_primeira_sessao', 'dia_primeira_sessao', 'mes_primeira_sessao', 'ano_primeira_sessao', 'dia_semana_primeira_sessao', 'data_ultima_sessao', 'dia_ultima_sessao', 'mes_ultima_sessao', 'ano_ultima_sessao', 'dia_semana_ultima_sessao', 'programa_codigo', 'programa_desc', 'atividade_codigo', 'atividade_desc', 'modalidade_codigo', 'modalidade_desc', 'realizacao_codigo', 'realizacao_desc', 'realizacao_nome', 'tipo_codigo', 'tipo_desc', 'subtipo_codigo', 'subtipo_desc', 'destino_turismo_desc', 'data_sessao', 'hora_sessao', 'dia_sessao', 'mes_sessao', 'ano_sessao', 'dia_semana_sessao', 'local_nome', 'grupo_local', 'acao_externa', 'lancamento_status', 'sessao_id', 'conjunto_id', 'conjunto_nome', 'grupo_id', 'grupo_nome', 'grupo_ordem', 'variavel_id

,uo,uor_sigla,atividade_id,atividade_nome,atividade_complemento,atividade_status,estatistico_status,projeto_id,projeto_nome,gratuidade,...,tipo_variavel_id,tipo_variavel_nome,valor_mensurador,eleito,ordem_linha,ordem_coluna,elegivel_pcg,mensurador_em_coluna,integrado,areaprogramatica_nome
0,53,SANTANA,5.300001e+13,Curumim - Turma 1 Tarde,Terca 14h Quinta 14h - De 7 a 12 anos,APROVADO,INTEGRADO,5.300000e+13,Curumim,1,...,10.0,PCG,0.0,0.0,1,1,N,Evasões no dia-Evasões,INTEGRADO,null
1,53,SANTANA,5.300001e+13,Curumim - Turma 1 Tarde,Terca 14h Quinta 14h - De 7 a 12 anos,APROVADO,INTEGRADO,5.300000e+13,Curumim,1,...,10.0,PCG,0.0,0.0,1,1,N,Evasões no dia-Evasões,INTEGRADO,null
2,53,SANTANA,5.300001e+13,Curumim - Turma 1 Tarde,Terca 14h Quinta 14h - De 7 a 12 anos,APROVADO,INTEGRADO,5.300000e+13,Curumim,1,...,10.0,PCG,0.0,0.0,1,1,N,Evasões no dia-Evasões,INTEGRADO,null
3,53,SANTANA,5.300001e+13,Curumim - Turma 1 Tarde,Terca 14h Quinta 14h - De 7 a 12 anos,APROVADO,INTEGRADO,5.300000e+13,Curumim,1,...,10.0,PCG,0.0,0.0,1,1,N,Evasões no dia-Evasões,INTEGRADO,null
4,53,SANTANA,5.300001e+13,Curumim - Turma 1 Tarde,Terca 14h Quinta 14h - De 7 a 12 anos,APROVADO,INTEGRADO,5.300000e+13,Curumim,1,...,10.0,PCG,0.0,0.0,1,1,N,Evasões no dia-Evasões,INTEGRADO,null


In [9]:
print(
#    df.groupby(["modalidade_desc", "realizacao_desc", "mensurador_em_coluna"])
    df.groupby(["mensurador_em_coluna"])
    .size()
    .rename("linhas")
    .sort_index()
    .to_string()
)

mensurador_em_coluna
-Acervo                                                  11671
-Acesso a conteúdos em domínios do Sesc                     12
-Arrecadação (kg)                                          241
-Audiência de TV                                             4
-Audiência de rádio                                          6
-Distribuição (kg)                                         241
-Diárias                                                   296
-Doadores Ativos                                           241
-Entidades Assistidas                                      241
-Equipes                                                  1199
-Exemplares de livros distribuídos                          14
-Exemplares de revistas distribuídas                         2
-Frequência                                               1213
-Inscritos                                                  93
-Instituições beneficiadas com livros distribuídos           2
-Instituições beneficiadas com rev

## Diagnóstico do RP (2024) — Subatividades × Serviços × Mensuradores

No RP (Referencial Programático, vigente até 2024) os campos `atividade_desc`, `modalidade_desc` e `realizacao_nome`
tinham valores distintos do RPS atual. Os qualificadores de público também eram diferentes:
`Comerciário`, `Dependente`, `Usuário` no lugar de `Pleno - Titular`, `Pleno - Dependente`, `MIS e Atividade`.

> **Pré-requisito:** `df` deve ter sido carregado com `ano_sessao = 2024` em `preview_100_rows`.

In [ ]:
print(
    df.groupby(["atividade_desc", "modalidade_desc", "realizacao_nome", "mensurador_em_coluna"])
    .size()
    .rename("linhas")
    .sort_index()
    .to_string()
)

## Mapeia principais mensuradores do RPS

In [ ]:
# Mapeia o siplan+lancamento para captar as variáveis-chave do RPS
MENS_MAP = {
    "Pessoas atendidas": "pessoas",
    "Presenças"        : "presentes",
    "Inscritos no dia" : "inscritos",
    "Evasões no dia"   : "evadidos",
    "Concluintes"      : "concluintes",
    ""                 : "numero",
}

df["_mens"] = df["mensurador_em_coluna"].str.split("-", n=1).str[0]
df_f = df[df["_mens"].isin(MENS_MAP)].copy()

for mens, col in MENS_MAP.items():
    df_f[col] = df_f["valor_mensurador"].where(df_f["_mens"] == mens, 0)

# Pleno
_pleno = df_f["mensurador_em_coluna"].str.contains("Pleno", na=False)
df_f["pessoas_plenos"]   = df_f["valor_mensurador"].where(_pleno & (df_f["_mens"] == "Pessoas atendidas"), 0)
df_f["presentes_plenos"] = df_f["valor_mensurador"].where(_pleno & (df_f["_mens"] == "Presenças"),         0)

# Presenças por tipo de acesso
_pres = df_f["_mens"] == "Presenças"
_tv   = df_f["tipo_variavel_nome"]
df_f["presentes_pagos"]     = df_f["valor_mensurador"].where(_pres & (_tv == "Pago"),     0)
df_f["presentes_gratuitos"] = df_f["valor_mensurador"].where(_pres & (_tv == "Gratuito"), 0)
df_f["presentes_pcg"]       = df_f["valor_mensurador"].where(_pres & (_tv == "PCG"),      0)

# Converte dia_sessao para date simples
df_f["dia_sessao"] = pd.to_datetime(df_f["dia_sessao"], errors="coerce").dt.date

colunas_num = list(MENS_MAP.values()) + [
    "pessoas_plenos", "presentes_plenos",
    "presentes_pagos", "presentes_gratuitos", "presentes_pcg",
]
df_sessao = df_f.groupby(["atividade_id", "sessao_id"], as_index=False)[colunas_num].sum()

dia = df_f[["sessao_id", "dia_sessao"]].drop_duplicates("sessao_id")
df_sessao = df_sessao.merge(dia, on="sessao_id", how="left")

print(f"Linhas originais  : {len(df):>10,}")
print(f"Linhas filtradas  : {len(df_f):>10,}")
print(f"Sessões únicas    : {len(df_sessao):>10,}")
print(f"Redução total     : {1 - len(df_sessao)/len(df):.1%}")
df_sessao.head(10)

## Quadro final por mês

In [58]:
colunas_num = list(MENS_MAP.values()) + ["pessoas_plenos", "presentes_plenos"]

df_mensal = (
    df
    .assign(ano_mes=pd.to_datetime(df["dia_sessao"]).dt.to_period("M"))
    .groupby("ano_mes")[colunas_num]
    .sum()
    .sort_index()
)

print(df_mensal.to_string())

           pessoas  presentes  inscritos  evadidos  concluintes    numero  pessoas_plenos  presentes_plenos
ano_mes                                                                                                    
2025-01  3545432.0  2735350.0   149153.0     834.0       1573.0   40707.0       1013730.0          859491.0
2025-02  2438268.0  1958955.0    69763.0    1117.0       2415.0   38990.0        798682.0          818907.0
2025-03  2760861.0  2169353.0    80251.0     877.0       1689.0   50250.0        836019.0          841365.0
2025-04  1977230.0  1549307.0    70677.0     895.0       3143.0   42613.0        539697.0          621130.0
2025-05  2528050.0  2054346.0   106201.0    1068.0       6944.0   60938.0        540849.0          639476.0
2025-06  1939352.0  1504434.0    64123.0    1030.0       6428.0   51499.0        373431.0          457318.0
2025-07  2731427.0  2088932.0    92714.0    1086.0       4627.0   43020.0        514683.0          510015.0
2025-08  2320829.0  1793333.

## Consulta esquemas e tabelas do lakehouse conectado

In [ ]:
# Lista todos os schemas e tabelas disponíveis no Lakehouse
cur = conn.cursor()
cur.execute("""
    SELECT TABLE_SCHEMA, TABLE_NAME, TABLE_TYPE
    FROM INFORMATION_SCHEMA.TABLES
    ORDER BY TABLE_SCHEMA, TABLE_NAME
""")
print(f"{"SCHEMA":<20} {"TABELA":<40} {"TIPO"}")
print("-" * 70)
for row in cur.fetchall():
    print(f"{row[0]:<20} {row[1]:<40} {row[2]}")